# Submission 2 — LightGBM with Depth-1 Aggregations + Stability Filtering

Extends Submission 1 with aggregated features from 4 depth-1 tables:
- `applprev_1` — previous loan applications (6.5M rows)
- `credit_bureau_a_1` — credit bureau A records (15.9M rows)
- `credit_bureau_b_1` — credit bureau B records
- `person_1` — person-level demographics

Key addition over Submission 1: **stability-first feature filtering** — features
that drift significantly over time (high week-to-week mean shift) are dropped
even if they are predictive, to improve the gini stability score.

In [ ]:
import polars as pl
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from scipy.stats import linregress
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA = '/kaggle/input/home-credit-credit-risk-model-stability/parquet_files'
OUTPUT = '/kaggle/working'

TRAIN_CUTOFF = 60  # weeks 0-60 train, 61-91 validate

## 1. Load depth-0 tables (same as Submission 1)

In [ ]:
base = pl.read_parquet(f'{DATA}/train/train_base.parquet')
static0 = pl.scan_parquet(f'{DATA}/train/train_static_0_*.parquet').collect()
static_cb = pl.read_parquet(f'{DATA}/train/train_static_cb_0.parquet')

test_base = pl.read_parquet(f'{DATA}/test/test_base.parquet')
test_static0 = pl.scan_parquet(f'{DATA}/test/test_static_0_*.parquet').collect()
test_static_cb = pl.read_parquet(f'{DATA}/test/test_static_cb_0.parquet')

print(f'Base train: {base.shape}, test: {test_base.shape}')

## 2. Aggregate depth-1 tables

For each depth-1 table we compute per-case_id aggregations:
- Numeric columns: mean, max, min, std, count of non-nulls
- Categorical columns: count of unique values

Using Polars lazy evaluation to keep memory usage low.

In [ ]:
def aggregate_table(path: str, table_name: str) -> pl.DataFrame:
    df = pl.scan_parquet(path).collect()
    print(f'{table_name}: {df.shape}')
    
    num_cols = [
        c for c in df.columns
        if c != 'case_id' and df[c].dtype in [
            pl.Float32, pl.Float64, pl.Int8, pl.Int16, pl.Int32, pl.Int64
        ]
    ]
    cat_cols = [
        c for c in df.columns
        if c != 'case_id' and df[c].dtype == pl.Utf8
    ]
    
    aggs = [pl.len().alias(f'{table_name}_count')]
    
    for col in num_cols:
        aggs += [
            pl.col(col).mean().alias(f'{table_name}_{col}_mean'),
            pl.col(col).max().alias(f'{table_name}_{col}_max'),
            pl.col(col).min().alias(f'{table_name}_{col}_min'),
            pl.col(col).std().alias(f'{table_name}_{col}_std'),
        ]
    
    for col in cat_cols:
        aggs.append(pl.col(col).n_unique().alias(f'{table_name}_{col}_nunique'))
    
    result = df.group_by('case_id').agg(aggs)
    print(f'  -> {result.shape} after aggregation')
    return result

In [ ]:
agg_applprev = aggregate_table(
    f'{DATA}/train/train_applprev_1_*.parquet', 'applprev'
)
agg_cb_a = aggregate_table(
    f'{DATA}/train/train_credit_bureau_a_1_*.parquet', 'cb_a'
)
agg_cb_b = aggregate_table(
    f'{DATA}/train/train_credit_bureau_b_1.parquet', 'cb_b'
)
agg_person = aggregate_table(
    f'{DATA}/train/train_person_1.parquet', 'person'
)

In [ ]:
# Same aggregations for test set
agg_applprev_test = aggregate_table(
    f'{DATA}/test/test_applprev_1_*.parquet', 'applprev'
)
agg_cb_a_test = aggregate_table(
    f'{DATA}/test/test_credit_bureau_a_1_*.parquet', 'cb_a'
)
agg_cb_b_test = aggregate_table(
    f'{DATA}/test/test_credit_bureau_b_1.parquet', 'cb_b'
)
agg_person_test = aggregate_table(
    f'{DATA}/test/test_person_1.parquet', 'person'
)

## 3. Join everything

In [ ]:
def build_dataset(base_df, static0_df, static_cb_df, aggs):
    df = (
        base_df
        .join(static0_df, on='case_id', how='left')
        .join(static_cb_df, on='case_id', how='left')
    )
    for agg in aggs:
        df = df.join(agg, on='case_id', how='left')
    return df

train_aggs = [agg_applprev, agg_cb_a, agg_cb_b, agg_person]
test_aggs = [agg_applprev_test, agg_cb_a_test, agg_cb_b_test, agg_person_test]

df = build_dataset(base, static0, static_cb, train_aggs)
test = build_dataset(test_base, test_static0, test_static_cb, test_aggs)

print(f'Train: {df.shape}')
print(f'Test:  {test.shape}')

## 4. Preprocessing

In [ ]:
DROP_COLS = ['case_id', 'date_decision', 'MONTH', 'WEEK_NUM', 'target']
TEST_DROP = ['case_id', 'date_decision', 'MONTH', 'WEEK_NUM']

def preprocess(df: pl.DataFrame, drop_cols: list) -> pl.DataFrame:
    date_cols = [c for c in df.columns if c.endswith('_D')]
    for col in date_cols:
        if col not in drop_cols:
            df = df.with_columns(
                pl.col(col).str.to_date(strict=False).cast(pl.Int32).alias(col)
            )
    cat_cols = [c for c in df.columns if df[c].dtype == pl.Utf8 and c not in drop_cols]
    for col in cat_cols:
        df = df.with_columns(
            pl.col(col).cast(pl.Categorical).to_physical().alias(col)
        )
    return df.drop([c for c in drop_cols if c in df.columns])

week_num = df['WEEK_NUM'].to_numpy()
y = df['target'].to_numpy()

X_df = preprocess(df, DROP_COLS)
X_test_df = preprocess(test, TEST_DROP)

feature_cols = [c for c in X_df.columns if c in X_test_df.columns]
X_df = X_df.select(feature_cols)
X_test_df = X_test_df.select(feature_cols)

print(f'Features before stability filter: {len(feature_cols)}')

## 5. Stability-first feature filtering

For each numeric feature, compute its mean per WEEK_NUM on the training set.
Features whose per-week mean drifts significantly over time are likely to
cause week-to-week instability in predictions — we drop the worst offenders.

Metric: coefficient of variation (std / |mean|) of weekly feature means.

In [ ]:
def feature_temporal_drift(X_df: pl.DataFrame, week_num: np.ndarray) -> pl.DataFrame:
    df_with_week = X_df.with_columns(pl.Series('WEEK_NUM', week_num))
    
    numeric_cols = [
        c for c in X_df.columns
        if X_df[c].dtype in [pl.Float32, pl.Float64, pl.Int32, pl.Int64, pl.Int16, pl.Int8]
    ]
    
    records = []
    for col in numeric_cols:
        weekly_means = (
            df_with_week
            .group_by('WEEK_NUM')
            .agg(pl.col(col).mean())
            .sort('WEEK_NUM')[col]
            .drop_nulls()
            .to_numpy()
        )
        if len(weekly_means) < 5 or np.abs(weekly_means.mean()) < 1e-9:
            records.append({'feature': col, 'drift': 0.0})
            continue
        cv = weekly_means.std() / np.abs(weekly_means.mean())
        records.append({'feature': col, 'drift': float(cv)})
    
    return pl.DataFrame(records).sort('drift', descending=True)

# Only use training portion for drift calculation (no leakage)
train_mask = week_num <= TRAIN_CUTOFF
drift_df = feature_temporal_drift(X_df.filter(pl.Series(train_mask)), week_num[train_mask])

print('Most temporally unstable features:')
print(drift_df.head(20))

In [ ]:
# Drop top 10% most drifting features
n_drop = max(1, int(len(drift_df) * 0.10))
unstable_features = set(drift_df.head(n_drop)['feature'].to_list())
stable_features = [c for c in feature_cols if c not in unstable_features]

print(f'Dropped {len(unstable_features)} unstable features')
print(f'Remaining features: {len(stable_features)}')

X = X_df.select(stable_features).to_numpy()
X_test = X_test_df.select(stable_features).to_numpy()

## 6. Gini stability metric

In [ ]:
def gini_stability(week_nums, targets, preds):
    weeks = np.unique(week_nums)
    weekly_gini = []
    for w in weeks:
        mask = week_nums == w
        if targets[mask].sum() == 0:
            continue
        gini = 2 * roc_auc_score(targets[mask], preds[mask]) - 1
        weekly_gini.append(gini)
    weekly_gini = np.array(weekly_gini)
    mean_gini = weekly_gini.mean()
    slope, intercept, *_ = linregress(np.arange(len(weekly_gini)), weekly_gini)
    residuals = weekly_gini - (intercept + slope * np.arange(len(weekly_gini)))
    score = mean_gini + 88.0 * min(0, slope) - 0.5 * residuals.std()
    return score, mean_gini, slope, residuals.std(), weekly_gini

## 7. Time-based train/val split

In [ ]:
val_mask = week_num > TRAIN_CUTOFF

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
week_val = week_num[val_mask]

print(f'Train: {X_train.shape[0]:,} rows ({y_train.mean():.3%} positive)')
print(f'Val:   {X_val.shape[0]:,} rows ({y_val.mean():.3%} positive)')

## 8. Train LightGBM

In [ ]:
params = {
    'objective': 'binary',
    'metric': 'auc',
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'scale_pos_weight': (y_train == 0).sum() / (y_train == 1).sum(),
    'n_jobs': -1,
    'verbose': -1,
    'random_state': 42,
}

model = lgb.LGBMClassifier(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

print(f'Best iteration: {model.best_iteration_}')

## 9. Evaluate

In [ ]:
val_preds = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, val_preds)
score, mean_gini, slope, resid_std, weekly_gini = gini_stability(week_val, y_val, val_preds)

print(f'Validation AUC:        {auc:.4f}')
print(f'Gini stability score:  {score:.4f}')
print(f'  Mean weekly Gini:    {mean_gini:.4f}')
print(f'  Trend slope:         {slope:.6f}')
print(f'  Residual std:        {resid_std:.4f}')

In [ ]:
valid_weeks = np.unique(week_val)
valid_weeks = [w for w in valid_weeks if y_val[week_val == w].sum() > 0]

plt.figure(figsize=(10, 4))
plt.plot(valid_weeks, weekly_gini, marker='o', linewidth=1.5, label='Weekly Gini')
plt.axhline(mean_gini, color='red', linestyle='--', label=f'Mean = {mean_gini:.3f}')
plt.xlabel('WEEK_NUM')
plt.ylabel('Gini')
plt.title('Per-week Gini — Submission 2 (depth-1 + stability filter)')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Feature importance

In [ ]:
importance = pl.DataFrame({
    'feature': stable_features,
    'importance': model.feature_importances_
}).sort('importance', descending=True)

print('Top 20 features:')
print(importance.head(20))

plt.figure(figsize=(10, 8))
top20 = importance.head(20)
plt.barh(top20['feature'].to_list()[::-1], top20['importance'].to_list()[::-1])
plt.xlabel('Importance (split count)')
plt.title('Top 20 features — Submission 2')
plt.tight_layout()
plt.show()

## 11. Retrain on full data & generate submission

In [ ]:
final_params = {**params, 'n_estimators': model.best_iteration_}
final_model = lgb.LGBMClassifier(**final_params)
final_model.fit(X, y)

test_preds = final_model.predict_proba(X_test)[:, 1]

submission = pl.DataFrame({
    'case_id': test_base['case_id'],
    'score': test_preds
})

submission.write_csv(f'{OUTPUT}/submission.csv')
print(f'Saved {len(submission):,} predictions')
print(submission.describe())